# Quantitative Trading Strategy Analysis

This notebook demonstrates how to use the quantitative trading framework for strategy development and backtesting.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from data.data_handler import DataHandler
from strategies.momentum_strategy import MomentumStrategy
from strategies.mean_reversion_strategy import MeanReversionStrategy
from backtesting.backtest_engine import BacktestEngine
from portfolio.portfolio_optimizer import PortfolioOptimizer
from utils.performance_metrics import PerformanceMetrics
from utils.plotting import PerformancePlotter

%matplotlib inline

## 1. Load Market Data

In [ ]:
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA']
start_date = '2020-01-01'
end_date = '2023-12-31'

data_handler = DataHandler()
data = data_handler.get_data(tickers, start_date, end_date)

print(f"Data shape: {data.shape}")
data.head()

## 2. Data Exploration

In [ ]:
stats = data_handler.calculate_statistics(data)

print("Annual Returns:")
print(stats['mean_returns'])
print("\nAnnual Volatility:")
print(stats['volatility'])
print("\nCorrelation Matrix:")
print(stats['correlation'])

In [ ]:
plotter = PerformancePlotter()
returns = data_handler.get_returns(data)
plotter.plot_correlation_matrix(returns, "Asset Correlation")
plt.show()

## 3. Momentum Strategy Backtest

In [ ]:
momentum_strategy = MomentumStrategy(lookback_period=20, top_n=3)
backtest_engine = BacktestEngine(initial_capital=100000)
momentum_results = backtest_engine.run(momentum_strategy, data)

backtest_engine.print_results(momentum_results)

In [ ]:
plotter.plot_portfolio_value(momentum_results['portfolio_series'])
plt.show()

plotter.plot_drawdown(momentum_results['portfolio_series'])
plt.show()

## 4. Mean Reversion Strategy Backtest

In [ ]:
mr_strategy = MeanReversionStrategy(lookback_period=20, entry_threshold=2.0)
backtest_engine2 = BacktestEngine(initial_capital=100000)
mr_results = backtest_engine2.run(mr_strategy, data)

backtest_engine2.print_results(mr_results)

## 5. Strategy Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

momentum_cum = (1 + momentum_results['returns_series']).cumprod() - 1
mr_cum = (1 + mr_results['returns_series']).cumprod() - 1

ax.plot(momentum_cum.index, momentum_cum * 100, label='Momentum', linewidth=2)
ax.plot(mr_cum.index, mr_cum * 100, label='Mean Reversion', linewidth=2)

ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Returns (%)')
ax.set_title('Strategy Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 6. Portfolio Optimization

In [ ]:
optimizer = PortfolioOptimizer(stats['mean_returns'], stats['cov_matrix'])

max_sharpe = optimizer.max_sharpe_ratio()
min_vol = optimizer.min_volatility()
risk_parity = optimizer.risk_parity()
equal_weight = optimizer.equal_weight()

print("Max Sharpe Ratio Portfolio:")
print(f"  Return: {max_sharpe['return']*100:.2f}%")
print(f"  Volatility: {max_sharpe['volatility']*100:.2f}%")
print(f"  Sharpe: {max_sharpe['sharpe_ratio']:.2f}")
print("  Weights:", max_sharpe['weights'].to_dict())

print("\nMin Volatility Portfolio:")
print(f"  Return: {min_vol['return']*100:.2f}%")
print(f"  Volatility: {min_vol['volatility']*100:.2f}%")
print(f"  Sharpe: {min_vol['sharpe_ratio']:.2f}")

In [ ]:
max_sharpe['weights'].plot(kind='bar', figsize=(10, 5))
plt.title('Max Sharpe Ratio Portfolio Weights')
plt.ylabel('Weight')
plt.xlabel('Asset')
plt.grid(True, alpha=0.3)
plt.show()

## 7. Performance Analysis

In [ ]:
metrics = PerformanceMetrics.calculate_all_metrics(momentum_results['returns_series'])
PerformanceMetrics.print_metrics(metrics)

In [ ]:
plotter.plot_rolling_metrics(momentum_results['returns_series'], window=60)
plt.show()